# Creating Synthetic Data - Regular and Anomalous Data

We create synthetic data when we lack real data. A model trained on merely synthetic data cannot replace a model trained on real data. Synthetic are useful in early projects/POCs/Demos etc.

Before creating synthetic data, we must ask ourselves three questions:
1. What is *normal behavior* in our system?
2. What *constraints* make it *normal*?
3. What *mechanisms* could plausibly break those constraints and thus, the normality?

## Normal data

These questions can be answered in a layered manner:
1. Structured constraints
    - Bounds: Bounds - max/min.
    - Relationships: A increases and B increases as well.
    - Conservation rules: Ratios in these increases.

2. Temporal coherence
    If the data is intended to be a time-series it will be wise to introduce the core characteristics of time-series data:
    - Seasonality
    - Trends
    etc.

3. Noise model  
    - Measurement models
    - Rounding
    - Missing Patterns

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Creating a basic normal data series

We are creating a basic normal industrial synthetic data series here. The idea is as following:
`ideal value + random deviation`

This is Gaussian Noise. In blunt terms, positive and negative errors occurs concentrated near the mean. The Below code generates such a dataset:

In [10]:
# The total number of records to be generated
num_records = 1000

# The start time for the generation
start_time = pd.Timestamp("2024-06-12 10:52:00")

# Duration between each timestamp intervals
interval = pd.Timedelta(minutes=10)

# Generate timestamps for num_records number of records
timestamps = [start_time + i * interval for i in range(num_records)]

# Makes the random numbers of to be the same on each run - deterministic
np.random.seed(42)

# Parameters, loc = mean, scale = standard deviation (the spread from the mean), size = independent draws
temperature = np.random.normal(loc=0.06, scale=0.002, size=num_records)
vibration = np.random.normal(loc=0.075, scale=0.002, size=num_records)
motor_rpm = np.random.normal(loc=0.045, scale=0.001, size=num_records)
motor_amps = np.random.normal(loc=0.084, scale=0.002, size=num_records)

df = pd.DataFrame()
df['timestamp'] = timestamps
df['temperature'] = temperature
df['vibration'] = vibration
df['motor_rpm'] = motor_rpm
df['motor_amps'] = motor_amps

df

,timestamp,temperature,vibration,motor_rpm,motor_amps
0,2024-06-12 10:52:00,0.060993,0.077799,0.044325,0.080184
1,2024-06-12 11:02:00,0.059723,0.076849,0.044855,0.082279
2,2024-06-12 11:12:00,0.061295,0.075119,0.044208,0.083173
3,2024-06-12 11:22:00,0.063046,0.073706,0.044692,0.087775
4,2024-06-12 11:32:00,0.059532,0.076396,0.043106,0.085113
...,...,...,...,...,...
995,2024-06-19 08:42:00,0.059438,0.077140,0.045077,0.084057
996,2024-06-19 08:52:00,0.063595,0.074947,0.045258,0.079844
997,2024-06-19 09:02:00,0.061282,0.073236,0.043758,0.083359
998,2024-06-19 09:12:00,0.058858,0.074674,0.045334,0.087287


The dataset thus generated is however quite basic. By this I mean that each variable is independent (which is generally not) and temporally coherence is not even introduced. We'll try doing this next. There are a few steps we can make for this:
1. Create smoother transitions between each day. (Temperature won't just randomly change with each instance, it would do so coherently).
2. Introduce dependencies between the variables. (Vibration and Temperature are coupled).
3. Add trends and seasonality. (Environmental effects, Motor warm up hrs etc.)